# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append(".")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface
from ase.visualize import view

import site_analysis as sa

#traj visualization
import mdtraj as md
import nglview as nv

In [ ]:
# USER INPUT
name = 'pd332.xyz'
n_layers = 10
atoms = surface('Pd',(3,3,2),n_layers) #lattice constant
atoms.center(vacuum=10, axis=2)
adsorbate_height = 2
site_bond_cutoff = 2

In [3]:
#visualize slab
write(name, atoms)
view(atoms, viewer='x3d')

In [4]:

# Load slab
slab = read(name)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [5]:
# rattled surface
slabrat = slab.copy()
slabrat.rattle(stdev=0.3)
view(slabrat, viewer='x3d')

# do not rattle the surface!!

In [6]:

slabrat = slab.copy()
slabrat.rattle(stdev=0.3)

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, "fcc332", composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()

#do not rattle the surface. it distorts the surface


There are 31 unique sites out of 72.


In [15]:

# Load the trajectory file

# Load the trajectory file
unique_sites = read('unique_sites.traj', index=1)
view(unique_sites, viewer='x3d')

In [ ]:
#test
cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)
single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()

/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (37, 118, 157)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (77, 117, 158)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (38, 77, 117)
/home/shikim/miniconda3/envs/pynta_env/lib/python3.9/site-packages/acat/adsorption_sites.py:2128: UserWarning: Cannot identify site (37, 78, 157)


There are 17 unique sites out of 22.


In [ ]:
print(cas.get_sites())

[{'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'subsurf', 'position': array([ 3.67473449,  1.7966486 , 18.27828973]), 'normal': nan, 'indices': (32, 35), 'composition': 'PdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([32, 35])}, {'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'subsurf', 'position': array([ 4.5671617 ,  2.84311226, 18.27828973]), 'normal': nan, 'indices': (32, 35), 'composition': 'PdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([32, 35])}, {'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'subsurf', 'position': array([ 2.78230728,  0.75018494, 18.27828973]), 'normal': nan, 'indices': (32, 35), 'composition': 'PdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([32, 35])}, {'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'subsurf', 'position': array([-11.49652799,   3.88957591,  18.27828973]), 'normal': na

In [ ]:
[cas.get_sites()[i]['morphology'] for i in range(len(cas.get_sites()))]

['subsurf',
 'subsurf',
 'subsurf',
 'subsurf',
 'terrace',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'sc-tc',
 'terrace',
 'terrace',
 'terrace',
 'terrace',
 'step',
 'step',
 'step',
 'sc-tc',
 'sc-tc',
 'step',
 'sc-tc',
 'terrace']

In [ ]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [ ]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [ ]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [ ]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [ ]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = True
0 vs 3 = False
0 vs 4 = False
1 vs 2 = False
1 vs 3 = False
1 vs 4 = False
2 vs 3 = False
2 vs 4 = False
3 vs 4 = False


In [ ]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 4
3-fold site type: [0, 2]
3-fold site type: [1]
3-fold site type: [3]
3-fold site type: [4]


In [ ]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 4 -> 3fold1
Geometry 7 -> 3fold1
Geometry 5 -> 3fold2
Geometry 9 -> 3fold3
Geometry 12 -> 3fold4


In [ ]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml
